# 🎯 Clase 3: Datos para Entrenamiento

**Curso: Modelos de AI en Negocios (MAIN) - IA Aplicada a la Industria**  
Universidad de Santiago de Chile · Departamento de Ingeniería Industrial

---

## 🔁 Recordemos dónde quedamos

En la **Clase 2** recibimos un dataset sucio de defectos de manufactura (1.000 registros) y lo dejamos limpio: corregimos fechas, rellenamos nulos y eliminamos costos imposibles.

Pero queda una pregunta abierta: **¿esos datos limpios ya son suficientes para que una IA aprenda de ellos?**

👉 **Todavía no.** Antes de poder usar inteligencia artificial sobre nuestros datos, hay que pasarlos por un **último proceso de transformación**. Eso es lo que veremos hoy.

---
## 🧠 Antes de empezar: ¿qué es realmente la "Inteligencia Artificial"?

Cuando hablamos de IA aplicada a la industria, en la mayoría de los casos nos referimos a **Machine Learning (ML)** — "aprendizaje automático".

### La idea central, en una frase

> Un programa tradicional sigue reglas escritas a mano. Un modelo de Machine Learning **descubre las reglas por sí mismo** a partir de muchos ejemplos.

### Comparación rápida

| | Programa tradicional | Modelo de Machine Learning |
|---|---|---|
| ¿Cómo funciona? | El programador escribe las reglas | El modelo aprende las reglas observando ejemplos |
| ¿Qué necesita? | Lógica + condiciones (if/else) | **Muchos ejemplos** (datos) |
| Ejemplo | "Si el costo > 500 → marcar como caro" | "Aquí tienes 1000 defectos pasados con su costo: aprende qué los hace caros" |

### ¿Y qué tienen que ver los datos con esto?

**Los datos son a la IA lo que los libros de texto son a un estudiante.** Si el material está desordenado, escrito en idiomas distintos o lleno de errores, el estudiante no aprende bien.

Hoy nuestro trabajo es preparar el "libro de texto" para que el modelo pueda estudiarlo correctamente.

---
## 🏭 El caso de hoy

Seguimos siendo analistas de la planta industrial. El equipo de calidad nos plantea una idea:

> *"¿Y si pudiéramos usar todos estos defectos pasados para que una IA aprenda a estimar el costo de reparación de un defecto futuro, antes de repararlo?"*

Esa idea (que veremos en clases siguientes) requiere un paso previo: **convertir nuestro CSV limpio en un formato que un modelo de IA pueda procesar**.

**Hoy NO entrenaremos ningún modelo todavía.** Hoy preparamos el material para que en las próximas clases podamos hacerlo sin sorpresas.

---
### Importar librerías

Como en la clase pasada, usamos `pandas`, `numpy` y `matplotlib`. Hoy agregamos una nueva librería que vamos a ver mucho de aquí en adelante:

| Librería | Apodo | ¿Para qué la usamos? |
|---|---|---|
| `pandas` | `pd` | Tablas (DataFrames) |
| `numpy` | `np` | Operaciones numéricas |
| `matplotlib.pyplot` | `plt` | Gráficos |
| **`scikit-learn`** | `sklearn` | **Herramientas de Machine Learning** (nuevo) |

> 💡 `scikit-learn` es la librería de IA más usada en la industria para datos tabulares. Trae listas las herramientas básicas para preparar datos y entrenar modelos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

print('✅ Librerías cargadas correctamente')

---
## 🗂️ Concepto 1: Datos tabulares vs. datos no estructurados

No todos los datos del mundo se ven igual. En la industria encontramos **dos grandes familias**:

### 📊 Datos tabulares (estructurados)

Son datos organizados en **filas y columnas**, como una tabla de Excel. Cada columna tiene un significado claro (fecha, cantidad, tipo, etc.).

**Ejemplos en la industria:**
- Registros de defectos de calidad ✅ *(es lo que tenemos)*
- Datos de ventas mensuales
- Lecturas de sensores con timestamp
- Inventario de la planta

### 🖼️ Datos no estructurados

Son datos que **no se acomodan a una tabla**. Cada "unidad" es un archivo complejo en sí mismo.

**Ejemplos en la industria:**
- Fotos de productos defectuosos (imágenes)
- Audio de máquinas funcionando (sonido)
- Reclamos de clientes escritos a mano (texto libre)
- Video de cámaras de seguridad

### Comparación rápida

| | Datos tabulares | Datos no estructurados |
|---|---|---|
| Formato | CSV, Excel, base SQL | Imágenes, audio, video, texto |
| Cada registro tiene… | Las mismas columnas que los demás | Una forma única |
| ¿Fácil de procesar? | Sí, casi listo para IA | Requiere modelos más complejos (Deep Learning) |
| Volumen típico | Miles a millones de filas | Cada archivo pesa más (MB cada uno) |

### 🎯 Idea clave

**Sea cual sea el tipo de dato, al final el modelo lo "ve" como una matriz de números.** Las imágenes se convierten en números (cada píxel es un número), el texto se convierte en números (cada palabra se mapea a un código), y los datos tabulares… también deben convertirse en números.

Hoy trabajaremos con **datos tabulares** (defectos de manufactura) y aprenderemos a convertirlos a una matriz puramente numérica.

---
## 📂 Paso 1: Cargar el dataset limpio

Cargamos el CSV original y aplicamos rápidamente las correcciones de la Clase 2 (en un proyecto real, leeríamos directamente el `defects_data_limpio.csv` que exportamos).

In [ ]:
df = pd.read_csv('defects_data.csv')
df['defect_date'] = pd.to_datetime(df['defect_date'])
df['severity'] = df['severity'].fillna('Desconocida')
df['inspection_method'] = df['inspection_method'].fillna('No registrado')
df = df[df['repair_cost'] >= 0].reset_index(drop=True)

print(f'Dataset limpio: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head()

---
## 🧠 Concepto 2: Aprender con ejemplos — la idea de "entrenar"

Cuando un humano aprende a reconocer una manzana, no lee un manual: ve **muchas manzanas** y poco a poco capta los patrones. Color, forma, tamaño.

Un modelo de IA hace algo similar: necesita **muchos ejemplos** del fenómeno que queremos que aprenda. Cada ejemplo tiene dos partes:

| Parte del ejemplo | ¿Qué es? | En nuestro caso (defectos) |
|---|---|---|
| **Características** (input) | La información que describe el caso | Tipo de defecto, ubicación, método de inspección |
| **Resultado** (output) | Lo que pasó realmente | Costo de reparación |

Si le mostramos al modelo **miles** de defectos pasados con sus características y su costo real, eventualmente el modelo aprende: 
*"Cuando un defecto es Estructural y se detecta en la zona Internal, suele costar más reparar."*

### Vocabulario que vamos a usar

| Término | Significado simple | Símbolo |
|---|---|---|
| **Características** o **features** | Las columnas que el modelo usa como pistas | `X` (mayúscula = matriz, varias columnas) |
| **Objetivo** o **target** | La columna que el modelo intenta adivinar | `y` (minúscula = una sola columna) |
| **Entrenar** | El proceso de mostrar los ejemplos al modelo para que aprenda | — |

> 💡 No confundir: el **objetivo** (`y`) es lo que el modelo va a **predecir**. Las **características** (`X`) son las pistas que le damos para que pueda predecirlo.

### Definir X (características) e y (objetivo) en nuestro caso

Para este ejercicio, supongamos que un futuro modelo querrá estimar el **costo de reparación** (`repair_cost`) a partir del resto de información del defecto. Entonces:

- **Objetivo (`y`):** la columna `repair_cost`
- **Características (`X`):** `defect_type`, `defect_location`, `severity`, `inspection_method`

**No usamos** `defect_id` ni `product_id` (son solo identificadores, sin información útil) ni `defect_date` (las fechas requieren un tratamiento especial que veremos más adelante).

In [ ]:
# Características (X): las columnas que serán "pistas" para el modelo
X = df[['defect_type', 'defect_location', 'severity', 'inspection_method']].copy()

# Objetivo (y): la columna que el modelo intentará adivinar
y = df['repair_cost'].copy()

print(f'X tiene {X.shape[0]} filas y {X.shape[1]} columnas (características)')
print(f'y tiene {y.shape[0]} valores (un costo por cada defecto)')
print()
print('Primeras 5 filas de X:')
print(X.head())
print()
print('Primeros 5 valores de y:')
print(y.head().tolist())

---
## 🔢 Concepto 3: Las máquinas no entienden palabras

Mira la primera fila de `X`. Algo así:

| defect_type | defect_location | severity | inspection_method |
|---|---|---|---|
| Structural | Surface | Critical | Visual Inspection |

**Problema:** un modelo de IA solo sabe trabajar con **números**. La palabra `"Critical"` no significa nada para él. Necesitamos una forma de traducir cada categoría a un número.

### ¿Cómo lo hacemos? Una columna por cada categoría posible

La técnica más usada se llama **One-Hot Encoding**. Suena complicada pero la idea es muy simple: por cada categoría posible, creamos una columna nueva que vale **1 si la fila pertenece a esa categoría, 0 si no**.

**Ejemplo con `defect_type`:**

| Antes | → | Después |
|---|---|---|
| `defect_type` | | `Structural` &nbsp;&nbsp; `Functional` &nbsp;&nbsp; `Cosmetic` |
| Structural | | **1** &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 0 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 0 |
| Functional | | 0 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **1** &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 0 |
| Cosmetic | | 0 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 0 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; **1** |

Ahora todo es **números** y el modelo puede trabajar con ello.

### ¿Por qué no asignar 1, 2, 3 directamente?

Sería tentador hacer: `Structural = 1`, `Functional = 2`, `Cosmetic = 3`. Pero esto **introduce un orden falso**: el modelo creería que `Cosmetic (3) > Functional (2) > Structural (1)`, como si fueran números reales. Eso no tiene sentido — son categorías sin orden.

Con One-Hot Encoding cada categoría es independiente. ✅

### En Python lo hace una sola línea

Pandas tiene la función `pd.get_dummies()` que aplica One-Hot Encoding automáticamente.

In [ ]:
X_numerico = pd.get_dummies(X, dtype=int)

print(f'Antes:  {X.shape[1]} columnas (con palabras)')
print(f'Después: {X_numerico.shape[1]} columnas (todo números)')
print()
print('Nuevas columnas:')
for col in X_numerico.columns:
    print(f'  - {col}')
print()
print('Así se ven las primeras filas ya en formato numérico:')
X_numerico.head()

Cada fila ahora es una secuencia de **0s y 1s**. Eso sí lo entiende un modelo de IA. 🎉

---
## 📏 Concepto 4: Cuando los números viven en escalas distintas

Imagina dos columnas dentro de los datos de un modelo:

| Columna | Rango de valores |
|---|---|
| `edad_de_la_maquina` | 1 a 30 (años) |
| `horas_de_uso_total` | 0 a 50.000 (horas) |

Para muchos modelos de IA, **la columna con números más grandes "pesa" más en las decisiones**, simplemente porque sus números son más grandes — aunque ambas variables sean igualmente importantes.

**Es como medir personas en metros y árboles en kilómetros y luego compararlos: el árbol parecerá insignificante solo por la unidad de medida.**

### La solución: poner todas las variables en la misma escala

Le pedimos a Python que **transforme todas las columnas numéricas a un rango común**, por ejemplo entre 0 y 1.

Esa técnica se llama **Min-Max Scaling** y básicamente hace:

$$ \text{valor escalado} = \frac{\text{valor} - \text{mínimo}}{\text{máximo} - \text{mínimo}} $$

Después de aplicarla, **todos los valores caen entre 0 y 1**, sin importar la escala original.

> 💡 En nuestro caso, X ya está toda en 0/1 después de One-Hot Encoding, así que el escalado no cambiaría mucho. Pero practicaremos la técnica sobre la columna **`y`** (`repair_cost`) para que vean cómo se hace y qué efecto tiene.

In [ ]:
# Mostramos repair_cost antes del escalado
print('repair_cost ANTES del escalado:')
print(f'  mínimo: {y.min():.2f}')
print(f'  máximo: {y.max():.2f}')
print(f'  promedio: {y.mean():.2f}')

Ahora aplicamos el escalado. La sintaxis tiene dos pasos:

1. `scaler = MinMaxScaler()` → creamos la herramienta
2. `scaler.fit_transform(...)` → la herramienta "mira" los datos y los transforma

> Como `MinMaxScaler` espera una matriz (no un vector), usamos `.values.reshape(-1, 1)` para darle el formato correcto.

In [ ]:
scaler = MinMaxScaler()
y_escalado = scaler.fit_transform(y.values.reshape(-1, 1)).flatten()

print('repair_cost DESPUÉS del escalado:')
print(f'  mínimo: {y_escalado.min():.2f}')
print(f'  máximo: {y_escalado.max():.2f}')
print(f'  promedio: {y_escalado.mean():.2f}')
print()
print('✅ Ahora todos los valores están entre 0 y 1.')
print('   El valor original más barato → 0.00')
print('   El valor original más caro   → 1.00')
print('   Todo lo demás queda en proporción intermedia.')

### Visualicemos el efecto del escalado

Los datos siguen teniendo la **misma forma** (la misma distribución), solo que ahora viven en un rango distinto. Es como traducir una temperatura de Celsius a Fahrenheit: la información es la misma, solo cambia la "escala" en la que la expresamos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y, bins=30, color='#1565C0', edgecolor='white')
axes[0].set_title('repair_cost ANTES (escala original)', fontweight='bold')
axes[0].set_xlabel('Costo de reparación ($)')
axes[0].set_ylabel('Cantidad de defectos')
axes[0].grid(axis='y', linestyle='--', alpha=0.4)
axes[0].spines[['top', 'right']].set_visible(False)

axes[1].hist(y_escalado, bins=30, color='#EA7600', edgecolor='white')
axes[1].set_title('repair_cost DESPUÉS (escala 0–1)', fontweight='bold')
axes[1].set_xlabel('Costo escalado')
axes[1].set_ylabel('Cantidad de defectos')
axes[1].grid(axis='y', linestyle='--', alpha=0.4)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_escalado.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Misma forma, distinta escala. El modelo verá los datos del lado derecho.')

---
## ✂️ Concepto 5: Datos para aprender vs. datos para evaluar

Imagina que un profesor enseña matemáticas usando 100 ejercicios resueltos, y luego evalúa al alumno **con esos mismos 100 ejercicios**. ¿Aprendió de verdad o solo memorizó las respuestas?

Lo mismo pasa con los modelos de IA. Si entrenamos el modelo con **todos los datos** y luego lo evaluamos con **esos mismos datos**, no sabremos si realmente aprendió.

### La solución: separar los datos en dos grupos antes de entrenar

| Grupo | ¿Para qué sirve? | Tamaño típico |
|---|---|---|
| **Datos de entrenamiento** | El modelo "estudia" con ellos | ~80% |
| **Datos de prueba** | El modelo nunca los ha visto. Sirven para evaluar si aprendió de verdad | ~20% |

### Diagrama de la idea

```
Dataset completo (100%):
├──────── ENTRENAMIENTO (80%) ────────┤── PRUEBA (20%) ──┤
          El modelo aprende aquí        Evaluamos aquí
```

### En Python lo hace una sola función

`train_test_split` de scikit-learn divide los datos al azar respetando el porcentaje que pidamos.

| Parámetro | Qué hace |
|---|---|
| `test_size=0.2` | El 20% va a prueba (el 80% queda para entrenar) |
| `random_state=42` | Fija la "semilla" del azar para que el resultado sea reproducible |

> 💡 **¿Por qué fijar `random_state`?** Porque el split usa una mezcla aleatoria. Si dos personas corren el mismo código sin fijar la semilla, obtendrán divisiones distintas. Fijarla garantiza que todos veamos el mismo resultado.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_numerico, y,
    test_size=0.2,
    random_state=42
)

print('=== División de los datos ===')
print(f'  Total                : {len(df)} filas')
print(f'  → Entrenamiento (80%): {len(X_train)} filas')
print(f'  → Prueba (20%)       : {len(X_test)} filas')
print()
print('Importante: el modelo aprenderá SOLO con X_train e y_train.')
print('X_test e y_test se guardan para evaluar al final, sin trampas.')

### Visualicemos la división

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.5))

ax.barh([''], [len(X_train)], color='#00a499', edgecolor='white', label=f'Entrenamiento ({len(X_train)} filas)')
ax.barh([''], [len(X_test)], left=[len(X_train)], color='#EA7600', edgecolor='white', label=f'Prueba ({len(X_test)} filas)')

ax.text(len(X_train) / 2, 0, f'80%\n{len(X_train)} filas', ha='center', va='center',
        color='white', fontweight='bold', fontsize=12)
ax.text(len(X_train) + len(X_test) / 2, 0, f'20%\n{len(X_test)} filas', ha='center', va='center',
        color='white', fontweight='bold', fontsize=12)

ax.set_title('División del dataset: Entrenamiento vs. Prueba', fontweight='bold', fontsize=12)
ax.set_xlim(0, len(df))
ax.set_xticks([])
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.4), ncol=2, frameon=False)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_split.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 💾 Paso final: Guardar los datos listos para entrenamiento

Exportamos los cuatro objetos finales como archivos CSV. **Estos archivos son la entrada del próximo paso del pipeline:** entrenar un modelo de IA (lo veremos en la Unidad 4).

| Archivo | ¿Qué contiene? |
|---|---|
| `X_train.csv` | Características de los defectos para que el modelo aprenda |
| `y_train.csv` | Costos reales correspondientes — las "respuestas correctas" para aprender |
| `X_test.csv` | Características que el modelo nunca verá durante el entrenamiento |
| `y_test.csv` | Costos reales para comparar con las predicciones del modelo en la evaluación |

In [ ]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print('✅ 4 archivos exportados:')
print('   - X_train.csv, y_train.csv  → para que el modelo aprenda')
print('   - X_test.csv,  y_test.csv   → para evaluar al modelo después')

---
## ✅ Conclusiones

### Lo que aprendimos hoy

1. **Datos limpios ≠ datos listos para IA.** Hay un paso de transformación intermedio que es obligatorio.
2. **Los modelos de IA solo entienden números.** Cualquier categoría debe traducirse a una representación numérica usando técnicas como One-Hot Encoding.
3. **Las escalas importan.** Cuando las columnas tienen rangos muy distintos, conviene normalizarlas para que ninguna domine sobre las otras.
4. **Hay que separar entrenamiento y prueba.** Si evaluamos al modelo con los datos que usó para aprender, no sabremos si realmente aprendió. Por eso reservamos un 20% "sin tocar" para la evaluación final.
5. **El pipeline siempre va en este orden:** datos sucios → limpieza (Clase 2) → preparación para entrenamiento (Clase 3) → entrenamiento del modelo (próxima unidad) → evaluación.

---

### Glosario rápido

| Término | Significado simple |
|---|---|
| **Machine Learning (ML)** | Tipo de IA donde el modelo aprende patrones a partir de muchos ejemplos |
| **Características / Features (X)** | Las columnas que el modelo usa como pistas |
| **Objetivo / Target (y)** | La columna que el modelo intenta adivinar |
| **Entrenar** | Mostrar ejemplos al modelo para que aprenda |
| **One-Hot Encoding** | Convertir una categoría de texto en columnas 0/1 |
| **Escalado / Normalización** | Llevar los números a un rango común (ej. 0 a 1) |
| **Datos de entrenamiento** | Los que usa el modelo para aprender |
| **Datos de prueba** | Los que reservamos para evaluar al modelo después |

---

### Próxima clase

En las próximas unidades veremos los **algoritmos de Machine Learning** propiamente dichos: cómo un modelo aprende a partir de los datos que preparamos hoy y qué tipos de problemas puede resolver (predecir cantidades, clasificar categorías, agrupar similitudes).